# Soccer Database – Exploratory Data Analysis (SQLite)

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('/Users/hasvardanyan/Downloads/soccer.sqlite')

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes


In [2]:
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

def run_query(sql):
    result = pd.read_sql(sql, conn)
    display(result)

## 1. Database overview

### Rows per table

In [3]:
query = """
SELECT
    (SELECT COUNT(*) FROM Country)           AS country,
    (SELECT COUNT(*) FROM League)            AS league,
    (SELECT COUNT(*) FROM Team)              AS team,
    (SELECT COUNT(*) FROM Team_Attributes)   AS team_attributes,
    (SELECT COUNT(*) FROM Player)            AS player,
    (SELECT COUNT(*) FROM Player_Attributes) AS player_attributes,
    (SELECT COUNT(*) FROM Match)             AS match
"""

run_query(query)

,country,league,team,team_attributes,player,player_attributes,match
0,11,11,299,1458,11060,183978,25979


Match has 25,979 rows and Player_Attributes has 183,978 rows; the database covers 299 teams and 11 leagues.

### Missing values

In [4]:
query = """
SELECT
    COUNT(*)                                   AS total_matches,
    COUNT(*) - COUNT(possession)               AS missing_possession,
    COUNT(*) - COUNT(shoton)                   AS missing_shots_on_target,
    COUNT(*) - COUNT(card)                     AS missing_cards,
    COUNT(*) - COUNT(B365H)                    AS missing_bet365_odds,
    COUNT(*) - COUNT(home_player_1)            AS missing_home_goalkeeper
FROM Match
"""

run_query(query)

,total_matches,missing_possession,missing_shots_on_target,missing_cards,missing_bet365_odds,missing_home_goalkeeper
0,25979,11762,11762,11762,3387,1224


In [5]:
query = """
SELECT
    COUNT(*)                                   AS total_rows,
    COUNT(*) - COUNT(overall_rating)           AS missing_overall_rating,
    COUNT(*) - COUNT(preferred_foot)           AS missing_preferred_foot,
    COUNT(*) - COUNT(volleys)                  AS missing_volleys,
    COUNT(*) - COUNT(gk_diving)                AS missing_gk_diving
FROM Player_Attributes
"""

run_query(query)

,total_rows,missing_overall_rating,missing_preferred_foot,missing_volleys,missing_gk_diving
0,183978,836,836,2713,836


About 45% of matches (11,762) have no possession, shots or cards data; Bet365 odds are missing in 13% of matches. Player ratings are almost complete (0.45% missing).

## 2. Leagues and goals

### Goals per match by league

In [6]:
query = """
SELECT
    Country.name                               AS country,
    League.name                                AS league,
    COUNT(Match.id)                            AS matches_played,
    COUNT(DISTINCT Match.season)               AS seasons,
    SUM(Match.home_team_goal + Match.away_team_goal) AS total_goals,
    ROUND(AVG(Match.home_team_goal + Match.away_team_goal), 2) AS goals_per_match
FROM Match
JOIN League  ON Match.league_id  = League.id
JOIN Country ON League.country_id = Country.id
GROUP BY Country.name, League.name
ORDER BY goals_per_match DESC
"""

run_query(query)

,country,league,matches_played,seasons,total_goals,goals_per_match
0,Netherlands,Netherlands Eredivisie,2448,8,7542,3.08
1,Switzerland,Switzerland Super League,1422,8,4166,2.93
2,Germany,Germany 1. Bundesliga,2448,8,7103,2.90
3,Belgium,Belgium Jupiler League,1728,8,4841,2.80
4,Spain,Spain LIGA BBVA,3040,8,8412,2.77
5,England,England Premier League,3040,8,8240,2.71
6,Scotland,Scotland Premier League,1824,8,4804,2.63
7,Italy,Italy Serie A,3017,8,7895,2.62
8,Portugal,Portugal Liga ZON Sagres,2052,8,5201,2.53
9,France,France Ligue 1,3040,8,7427,2.44


The Netherlands Eredivisie has the most goals per match (3.08); Poland (2.42) and France (2.44) have the fewest.

### Home advantage by league

In [7]:
query = """
SELECT
    League.name AS league,
    COUNT(*) AS matches,
    ROUND(100.0 * SUM(Match.home_team_goal > Match.away_team_goal) / COUNT(*), 1) AS home_win_pct,
    ROUND(100.0 * SUM(Match.home_team_goal = Match.away_team_goal) / COUNT(*), 1) AS draw_pct,
    ROUND(100.0 * SUM(Match.home_team_goal < Match.away_team_goal) / COUNT(*), 1) AS away_win_pct
FROM Match
JOIN League ON Match.league_id = League.id
GROUP BY League.name
ORDER BY home_win_pct DESC
"""

run_query(query)

,league,matches,home_win_pct,draw_pct,away_win_pct
0,Spain LIGA BBVA,3040,48.8,23.2,28.0
1,Netherlands Eredivisie,2448,47.8,23.7,28.4
2,Belgium Jupiler League,1728,46.9,24.6,28.5
3,Italy Serie A,3017,46.6,26.4,27.0
4,Switzerland Super League,1422,45.7,24.3,30.0
5,England Premier League,3040,45.7,25.8,28.5
6,Poland Ekstraklasa,1920,45.3,27.3,27.3
7,Germany 1. Bundesliga,2448,45.2,24.4,30.4
8,France Ligue 1,3040,44.7,28.3,27.0
9,Portugal Liga ZON Sagres,2052,44.2,26.0,29.8


Overall the home team wins 45.9% of matches, with 25.4% draws and 28.7% away wins. Home advantage is largest in Spain (48.8% home wins vs 28.0% away) and smallest in Scotland (41.7% vs 33.8%).

### Goals per season

In [8]:
query = """
SELECT
    Match.season,
    COUNT(*) AS matches,
    ROUND(AVG(Match.home_team_goal), 2) AS avg_home_goals,
    ROUND(AVG(Match.away_team_goal), 2) AS avg_away_goals,
    ROUND(AVG(Match.home_team_goal + Match.away_team_goal), 2) AS avg_total_goals
FROM Match
JOIN League ON Match.league_id = League.id
GROUP BY Match.season
ORDER BY Match.season
"""

run_query(query)

,season,matches,avg_home_goals,avg_away_goals,avg_total_goals
0,2008/2009,3326,1.51,1.10,2.61
1,2009/2010,3230,1.54,1.13,2.67
2,2010/2011,3260,1.55,1.14,2.68
3,2011/2012,3220,1.57,1.14,2.72
4,2012/2013,3260,1.55,1.22,2.77
5,2013/2014,3032,1.58,1.19,2.77
6,2014/2015,3325,1.52,1.16,2.68
7,2015/2016,3326,1.54,1.21,2.75


Total goals per match rose from 2.61 to 2.75 and away goals rose from 1.10 to 1.21, so the home-goal advantage fell from 0.41 to 0.33.

### Goals per month

In [9]:
query = """
SELECT
    strftime('%m', Match.date) AS month,
    COUNT(*) AS matches,
    ROUND(AVG(Match.home_team_goal + Match.away_team_goal), 2) AS goals_per_match
FROM Match
JOIN League ON Match.league_id = League.id
GROUP BY month
ORDER BY month
"""

run_query(query)

,month,matches,goals_per_match
0,01,2183,2.67
1,02,2834,2.63
2,03,2785,2.62
3,04,2969,2.74
4,05,2245,2.85
5,06,23,3.65
6,07,218,2.73
7,08,2276,2.63
8,09,2575,2.74
9,10,2608,2.73


May has the most goals per match (2.85); March, February and August have the fewest (about 2.62-2.63). June has only 23 matches.

## 3. Teams and matches

### Most one-sided matches

In [10]:
query = """
SELECT
    League.name AS league,
    Match.season,
    substr(Match.date, 1, 10) AS match_date,
    HomeTeam.team_long_name AS home_team,
    AwayTeam.team_long_name AS away_team,
    Match.home_team_goal || ' - ' || Match.away_team_goal AS score,
    ABS(Match.home_team_goal - Match.away_team_goal) AS goal_difference
FROM Match
JOIN League ON Match.league_id = League.id
JOIN Team AS HomeTeam ON Match.home_team_api_id = HomeTeam.team_api_id
JOIN Team AS AwayTeam ON Match.away_team_api_id = AwayTeam.team_api_id
ORDER BY goal_difference DESC, (Match.home_team_goal + Match.away_team_goal) DESC
LIMIT 10
"""

run_query(query)

,league,season,match_date,home_team,away_team,score,goal_difference
0,Netherlands Eredivisie,2010/2011,2010-10-24,PSV,Feyenoord,10 - 0,10
1,France Ligue 1,2015/2016,2016-03-13,ES Troyes AC,Paris Saint-Germain,0 - 9,9
2,Scotland Premier League,2010/2011,2010-11-06,Celtic,Aberdeen,9 - 0,9
3,Spain LIGA BBVA,2015/2016,2015-12-20,Real Madrid CF,Rayo Vallecano,10 - 2,8
4,England Premier League,2009/2010,2009-11-22,Tottenham Hotspur,Wigan Athletic,9 - 1,8
5,Spain LIGA BBVA,2014/2015,2015-04-05,Real Madrid CF,Granada CF,9 - 1,8
6,England Premier League,2009/2010,2010-05-09,Chelsea,Wigan Athletic,8 - 0,8
7,England Premier League,2012/2013,2012-12-23,Chelsea,Aston Villa,8 - 0,8
8,England Premier League,2014/2015,2014-10-18,Southampton,Sunderland,8 - 0,8
9,Germany 1. Bundesliga,2014/2015,2015-02-14,FC Bayern Munich,Hamburger SV,8 - 0,8


The biggest win is PSV 10-0 Feyenoord; the largest away win is Paris Saint-Germain 9-0 at Troyes; Real Madrid's 10-2 against Rayo Vallecano has the most goals (12).

### Total points per team (all seasons)

In [11]:
query = """
SELECT
    Team.team_long_name AS team,
    home.matches + away.matches AS matches,
    home.points + away.points AS total_points,
    ROUND(1.0 * (home.points + away.points) / (home.matches + away.matches), 2) AS points_per_match,
    home.goals_for + away.goals_for AS goals_scored,
    home.goals_against + away.goals_against AS goals_conceded
FROM Team
JOIN (
    SELECT home_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (home_team_goal > away_team_goal) + (home_team_goal = away_team_goal)) AS points,
           SUM(home_team_goal) AS goals_for,
           SUM(away_team_goal) AS goals_against
    FROM Match
    GROUP BY home_team_api_id
) AS home ON Team.team_api_id = home.team_id
JOIN (
    SELECT away_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (away_team_goal > home_team_goal) + (away_team_goal = home_team_goal)) AS points,
           SUM(away_team_goal) AS goals_for,
           SUM(home_team_goal) AS goals_against
    FROM Match
    GROUP BY away_team_api_id
) AS away ON Team.team_api_id = away.team_id
ORDER BY total_points DESC
LIMIT 10
"""

run_query(query)

,team,matches,total_points,points_per_match,goals_scored,goals_conceded
0,FC Barcelona,304,745,2.45,849,232
1,Real Madrid CF,304,720,2.37,843,304
2,Celtic,304,704,2.32,695,223
3,Juventus,301,633,2.10,543,250
4,Manchester United,304,633,2.08,582,280
5,FC Bayern Munich,272,623,2.29,653,211
6,FC Basel,286,604,2.11,619,311
7,Ajax,272,602,2.21,647,236
8,Paris Saint-Germain,304,601,1.98,568,267
9,Chelsea,304,598,1.97,583,286


FC Barcelona (745 points) and Real Madrid (720) are the most successful teams. Barcelona scored 849 goals and conceded 232 in 304 matches.

### Best team seasons by points

In [12]:
query = """
SELECT
    Team.team_long_name AS team,
    League.name AS league,
    home.season,
    home.matches + away.matches AS matches,
    home.points + away.points AS total_points
FROM Team
JOIN (
    SELECT league_id, season, home_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (home_team_goal > away_team_goal) + (home_team_goal = away_team_goal)) AS points
    FROM Match
    GROUP BY league_id, season, home_team_api_id
) AS home ON Team.team_api_id = home.team_id
JOIN (
    SELECT league_id, season, away_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (away_team_goal > home_team_goal) + (away_team_goal = home_team_goal)) AS points
    FROM Match
    GROUP BY league_id, season, away_team_api_id
) AS away ON home.team_id = away.team_id AND home.league_id = away.league_id AND home.season = away.season
JOIN League ON home.league_id = League.id
ORDER BY total_points DESC
LIMIT 10
"""

run_query(query)

,team,league,season,matches,total_points
0,Juventus,Italy Serie A,2013/2014,38,102
1,Real Madrid CF,Spain LIGA BBVA,2011/2012,38,100
2,FC Barcelona,Spain LIGA BBVA,2012/2013,38,100
3,Celtic,Scotland Premier League,2013/2014,38,99
4,FC Barcelona,Spain LIGA BBVA,2009/2010,38,99
5,Paris Saint-Germain,France Ligue 1,2015/2016,38,96
6,Real Madrid CF,Spain LIGA BBVA,2009/2010,38,96
7,FC Barcelona,Spain LIGA BBVA,2010/2011,38,96
8,FC Barcelona,Spain LIGA BBVA,2014/2015,38,94
9,Rangers,Scotland Premier League,2010/2011,38,93


Juventus' 102 points in 2013/14 is the highest season total. Real Madrid and Barcelona hold 6 of the top 10 seasons (100 points each in 2011/12 and 2012/13).

### Premier League table 2015/2016

In [13]:
query = """
SELECT
    Team.team_long_name AS team,
    home.matches + away.matches AS matches,
    home.points + away.points AS total_points,
    home.goals_for + away.goals_for AS goals_scored,
    home.goals_against + away.goals_against AS goals_conceded
FROM Team
JOIN (
    SELECT home_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (home_team_goal > away_team_goal) + (home_team_goal = away_team_goal)) AS points,
           SUM(home_team_goal) AS goals_for,
           SUM(away_team_goal) AS goals_against
    FROM Match
    WHERE league_id = (SELECT id FROM League WHERE name = 'England Premier League')
      AND season = '2015/2016'
    GROUP BY home_team_api_id
) AS home ON Team.team_api_id = home.team_id
JOIN (
    SELECT away_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(3 * (away_team_goal > home_team_goal) + (away_team_goal = home_team_goal)) AS points,
           SUM(away_team_goal) AS goals_for,
           SUM(home_team_goal) AS goals_against
    FROM Match
    WHERE league_id = (SELECT id FROM League WHERE name = 'England Premier League')
      AND season = '2015/2016'
    GROUP BY away_team_api_id
) AS away ON Team.team_api_id = away.team_id
ORDER BY total_points DESC
LIMIT 10
"""

run_query(query)

,team,matches,total_points,goals_scored,goals_conceded
0,Leicester City,38,81,68,36
1,Arsenal,38,71,65,36
2,Tottenham Hotspur,38,70,69,35
3,Manchester City,38,66,71,41
4,Manchester United,38,66,49,35
5,Southampton,38,63,59,41
6,West Ham United,38,62,65,51
7,Liverpool,38,60,63,50
8,Stoke City,38,51,41,55
9,Chelsea,38,50,59,53


Leicester City won the 2015/16 Premier League with 81 points, 10 more than Arsenal (71). Chelsea finished 10th with 50.

### Best attacking seasons

In [14]:
query = """
SELECT
    Team.team_long_name AS team,
    League.name AS league,
    home.season,
    home.goals + away.goals AS goals_scored,
    home.matches + away.matches AS matches,
    ROUND(1.0 * (home.goals + away.goals) / (home.matches + away.matches), 2) AS goals_per_match
FROM Team
JOIN (
    SELECT league_id, season, home_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(home_team_goal) AS goals
    FROM Match
    GROUP BY league_id, season, home_team_api_id
) AS home ON Team.team_api_id = home.team_id
JOIN (
    SELECT league_id, season, away_team_api_id AS team_id,
           COUNT(*) AS matches,
           SUM(away_team_goal) AS goals
    FROM Match
    GROUP BY league_id, season, away_team_api_id
) AS away ON home.team_id = away.team_id AND home.league_id = away.league_id AND home.season = away.season
JOIN League ON home.league_id = League.id
ORDER BY goals_scored DESC
LIMIT 10
"""

run_query(query)

,team,league,season,goals_scored,matches,goals_per_match
0,Real Madrid CF,Spain LIGA BBVA,2011/2012,121,38,3.18
1,Real Madrid CF,Spain LIGA BBVA,2014/2015,118,38,3.11
2,FC Barcelona,Spain LIGA BBVA,2012/2013,115,38,3.03
3,FC Barcelona,Spain LIGA BBVA,2011/2012,114,38,3.00
4,FC Barcelona,Spain LIGA BBVA,2015/2016,112,38,2.95
5,FC Barcelona,Spain LIGA BBVA,2014/2015,110,38,2.89
6,Real Madrid CF,Spain LIGA BBVA,2015/2016,110,38,2.89
7,Ajax,Netherlands Eredivisie,2009/2010,106,34,3.12
8,FC Barcelona,Spain LIGA BBVA,2008/2009,105,38,2.76
9,Real Madrid CF,Spain LIGA BBVA,2013/2014,104,38,2.74


9 of the 10 best attacking seasons belong to Real Madrid or Barcelona; Real Madrid scored 121 goals in 2011/12. The exception is Ajax in 2009/10 (106 goals in 34 matches).

### Home vs away points per match (teams with at least 100 home matches)

In [15]:
query = """
SELECT
    Team.team_long_name AS team,
    ROUND(1.0 * home.home_points / home.home_matches, 2) AS home_points_per_match,
    ROUND(1.0 * away.away_points / away.away_matches, 2) AS away_points_per_match,
    ROUND(1.0 * home.home_points / home.home_matches - 1.0 * away.away_points / away.away_matches, 2) AS home_advantage
FROM Team
JOIN (
    SELECT home_team_api_id AS team_id,
           COUNT(*) AS home_matches,
           SUM(3 * (home_team_goal > away_team_goal) + (home_team_goal = away_team_goal)) AS home_points
    FROM Match
    GROUP BY home_team_api_id
) AS home ON Team.team_api_id = home.team_id
JOIN (
    SELECT away_team_api_id AS team_id,
           COUNT(*) AS away_matches,
           SUM(3 * (away_team_goal > home_team_goal) + (away_team_goal = home_team_goal)) AS away_points
    FROM Match
    GROUP BY away_team_api_id
) AS away ON Team.team_api_id = away.team_id
WHERE home.home_matches >= 100
ORDER BY home_advantage DESC
LIMIT 10
"""

run_query(query)

,team,home_points_per_match,away_points_per_match,home_advantage
0,Catania,1.75,0.59,1.16
1,Palermo,1.74,0.83,0.91
2,Jagiellonia Białystok,1.80,0.91,0.89
3,Valenciennes FC,1.59,0.73,0.86
4,Villarreal CF,1.96,1.10,0.86
5,Manchester City,2.35,1.51,0.84
6,Genoa,1.73,0.90,0.83
7,CA Osasuna,1.57,0.75,0.82
8,KV Kortrijk,1.80,0.99,0.81
9,FC Sochaux-Montbéliard,1.55,0.76,0.79


Catania has the biggest gap: 1.75 points per home match against 0.59 away.

## 4. Players

### Best players by average rating

In [16]:
query = """
SELECT
    Player.player_name,
    ROUND(AVG(Player_Attributes.overall_rating), 1) AS avg_rating,
    MAX(Player_Attributes.overall_rating) AS best_rating,
    COUNT(*) AS records
FROM Player_Attributes
JOIN Player ON Player_Attributes.player_api_id = Player.player_api_id
GROUP BY Player.player_api_id, Player.player_name
HAVING COUNT(*) >= 5
ORDER BY avg_rating DESC
LIMIT 10
"""

run_query(query)

,player_name,avg_rating,best_rating,records
0,Lionel Messi,92.2,94,26
1,Cristiano Ronaldo,91.3,93,25
2,Franck Ribery,88.5,90,24
3,Andres Iniesta,88.3,91,25
4,Zlatan Ibrahimovic,88.3,90,21
5,Arjen Robben,87.8,90,25
6,Xavi Hernandez,87.6,92,22
7,Wayne Rooney,87.2,93,36
8,Iker Casillas,87.0,91,22
9,Philipp Lahm,86.7,87,30


Lionel Messi (92.2) and Cristiano Ronaldo (91.3) are clearly ahead of the rest.

### Rating by age

In [17]:
query = """
SELECT
    strftime('%Y', Player_Attributes.date) - strftime('%Y', Player.birthday) AS age,
    COUNT(*) AS records,
    ROUND(AVG(Player_Attributes.overall_rating), 1) AS avg_overall,
    ROUND(AVG(Player_Attributes.potential), 1) AS avg_potential
FROM Player_Attributes
JOIN Player ON Player_Attributes.player_api_id = Player.player_api_id
GROUP BY age
HAVING records >= 500
ORDER BY age
"""

run_query(query)

,age,records,avg_overall,avg_potential
0,14,529,59.9,71.1
1,15,646,59.5,70.7
2,16,718,59.0,70.2
3,17,1262,58.7,71.6
4,18,3358,59.8,73.2
5,19,6941,61.8,73.8
6,20,9900,63.7,74.2
7,21,11865,65.7,74.7
8,22,12944,67.2,74.8
9,23,13770,68.2,74.7


Average rating rises from 58.7 at age 17 to about 71.2 at age 30 and then stays flat. The gap between potential and overall rating is 13.4 at age 18 and 1.0 at age 30.

### Preferred foot

In [18]:
query = """
SELECT
    Player_Attributes.preferred_foot AS foot,
    COUNT(DISTINCT Player.player_api_id) AS players,
    ROUND(AVG(Player_Attributes.overall_rating), 2) AS avg_overall,
    ROUND(AVG(Player_Attributes.finishing), 1) AS avg_finishing,
    ROUND(AVG(Player_Attributes.crossing), 1) AS avg_crossing,
    ROUND(AVG(Player_Attributes.dribbling), 1) AS avg_dribbling
FROM Player_Attributes
JOIN Player ON Player_Attributes.player_api_id = Player.player_api_id
WHERE Player_Attributes.preferred_foot IS NOT NULL
GROUP BY Player_Attributes.preferred_foot
"""

run_query(query)

,foot,players,avg_overall,avg_finishing,avg_crossing,avg_dribbling
0,left,3202,68.63,49.5,60.4,61.8
1,right,8979,68.59,50.0,53.4,58.3


About 74% of players are right-footed. Both groups have the same average rating (68.6), but left-footed players are better at crossing (60.4 vs 53.4) and dribbling (61.8 vs 58.3).

### Goalkeepers vs outfield players

In [19]:
query = """
SELECT
    Player_Attributes.gk_reflexes >= 60 AS is_goalkeeper,
    COUNT(DISTINCT Player.player_api_id) AS players,
    ROUND(AVG(Player.height), 1) AS avg_height_cm,
    ROUND(AVG(Player.weight) * 0.453592, 1) AS avg_weight_kg,
    ROUND(AVG(Player_Attributes.jumping), 1) AS avg_jumping,
    ROUND(AVG(Player_Attributes.sprint_speed), 1) AS avg_sprint_speed
FROM Player_Attributes
JOIN Player ON Player_Attributes.player_api_id = Player.player_api_id
WHERE Player_Attributes.gk_reflexes IS NOT NULL
GROUP BY is_goalkeeper
ORDER BY is_goalkeeper DESC
"""

run_query(query)

,is_goalkeeper,players,avg_height_cm,avg_weight_kg,avg_jumping,avg_sprint_speed
0,1,927,188.9,84.0,65.5,46.4
1,0,10373,181.3,75.9,67.1,69.8


Goalkeepers are on average 7.6 cm taller (188.9 vs 181.3 cm) and about 8 kg heavier, but much slower (sprint speed 46.4 vs 69.8).

### Young players with the largest growth room

In [20]:
query = """
SELECT
    Player.player_name,
    substr(Player.birthday, 1, 10) AS birthday,
    Player_Attributes.overall_rating AS overall,
    Player_Attributes.potential,
    Player_Attributes.potential - Player_Attributes.overall_rating AS growth_room
FROM Player_Attributes
JOIN Player ON Player_Attributes.player_api_id = Player.player_api_id
JOIN (
    SELECT player_api_id, MAX(date) AS latest_date
    FROM Player_Attributes
    GROUP BY player_api_id
) AS latest
  ON Player_Attributes.player_api_id = latest.player_api_id
 AND Player_Attributes.date = latest.latest_date
WHERE Player.birthday >= '1995-01-01'
ORDER BY growth_room DESC, Player_Attributes.potential DESC
LIMIT 10
"""

run_query(query)

,player_name,birthday,overall,potential,growth_room
0,Tom Davies,1998-06-30,51,73,22
1,Kylian Mbappe Lottin,1998-12-20,65,85,20
2,Reece Oxford,1998-12-16,65,85,20
3,Harry Souttar,1998-10-22,57,77,20
4,Jonathan Leko,1999-04-24,65,84,19
5,Felix Passlack,1998-05-29,62,81,19
6,Christopher Teichmann,1995-10-13,56,75,19
7,Kamil Jozwiak,1998-04-22,56,75,19
8,Silvan Hefti,1997-10-25,53,72,19
9,Sean McKirdy,1998-04-12,52,71,19


Tom Davies has the largest gap between potential and overall rating (+22); Kylian Mbappe and Reece Oxford are rated 65 with potential 85.

### Goalkeepers with the fewest goals conceded at home

In [21]:
query = """
SELECT
    Player.player_name,
    COUNT(*) AS matches_as_home_goalkeeper,
    ROUND(AVG(Match.away_team_goal), 2) AS avg_goals_conceded
FROM Match
JOIN Player ON Match.home_player_1 = Player.player_api_id
GROUP BY Player.player_api_id, Player.player_name
HAVING COUNT(*) >= 100
ORDER BY avg_goals_conceded ASC
LIMIT 10
"""

run_query(query)

,player_name,matches_as_home_goalkeeper,avg_goals_conceded
0,Victor Valdes,100,0.65
1,Gianluigi Buffon,112,0.69
2,Manuel Neuer,127,0.72
3,Petr Cech,126,0.72
4,Salvatore Sirigu,107,0.73
5,Pepe Reina,124,0.73
6,Rui Patricio,118,0.76
7,Joe Hart,138,0.77
8,Roman Weidenfeller,108,0.79
9,Morgan De Sanctis,111,0.82


Victor Valdes (0.65 goals per home match), Gianluigi Buffon (0.69) and Manuel Neuer (0.72) lead the list.

## 5. Team playing style and points

### Chance creation passing and home points

In [22]:
query = """
SELECT
    Team_Attributes.chanceCreationPassingClass AS style_class,
    COUNT(DISTINCT Match.home_team_api_id) AS teams,
    COUNT(*) AS home_matches,
    ROUND(AVG(3 * (Match.home_team_goal > Match.away_team_goal) + (Match.home_team_goal = Match.away_team_goal)), 3) AS avg_home_points
FROM Match
JOIN Team_Attributes ON Match.home_team_api_id = Team_Attributes.team_api_id
JOIN (
    SELECT team_api_id, MAX(date) AS latest_date
    FROM Team_Attributes
    GROUP BY team_api_id
) AS latest
  ON Team_Attributes.team_api_id = latest.team_api_id
 AND Team_Attributes.date = latest.latest_date
GROUP BY Team_Attributes.chanceCreationPassingClass
"""

run_query(query)

,style_class,teams,home_matches,avg_home_points
0,Normal,257,22510,1.621
1,Risky,22,2524,1.750
2,Safe,9,784,1.622


Teams with risky chance creation average 1.75 points per home match, against 1.62 for normal and safe teams.

### Defence pressure and home points

In [23]:
query = """
SELECT
    Team_Attributes.defencePressureClass AS style_class,
    COUNT(DISTINCT Match.home_team_api_id) AS teams,
    COUNT(*) AS home_matches,
    ROUND(AVG(3 * (Match.home_team_goal > Match.away_team_goal) + (Match.home_team_goal = Match.away_team_goal)), 3) AS avg_home_points
FROM Match
JOIN Team_Attributes ON Match.home_team_api_id = Team_Attributes.team_api_id
JOIN (
    SELECT team_api_id, MAX(date) AS latest_date
    FROM Team_Attributes
    GROUP BY team_api_id
) AS latest
  ON Team_Attributes.team_api_id = latest.team_api_id
 AND Team_Attributes.date = latest.latest_date
GROUP BY Team_Attributes.defencePressureClass
"""

run_query(query)

,style_class,teams,home_matches,avg_home_points
0,Deep,25,1775,1.434
1,High,2,166,2.283
2,Medium,261,23877,1.644


**Finding:** Deep defence teams average only 1.43 points per home match against 1.64 for medium pressure. Only 2 teams have high pressure, so that group is too small to trust.

### Build-up speed and home points

In [24]:
query = """
SELECT
    Team_Attributes.buildUpPlaySpeedClass AS style_class,
    COUNT(DISTINCT Match.home_team_api_id) AS teams,
    COUNT(*) AS home_matches,
    ROUND(AVG(3 * (Match.home_team_goal > Match.away_team_goal) + (Match.home_team_goal = Match.away_team_goal)), 3) AS avg_home_points
FROM Match
JOIN Team_Attributes ON Match.home_team_api_id = Team_Attributes.team_api_id
JOIN (
    SELECT team_api_id, MAX(date) AS latest_date
    FROM Team_Attributes
    GROUP BY team_api_id
) AS latest
  ON Team_Attributes.team_api_id = latest.team_api_id
 AND Team_Attributes.date = latest.latest_date
GROUP BY Team_Attributes.buildUpPlaySpeedClass
"""

run_query(query)

,style_class,teams,home_matches,avg_home_points
0,Balanced,259,22946,1.614
1,Fast,22,2314,1.840
2,Slow,7,558,1.552


**Finding:** Fast build-up teams average 1.84 points per home match (balanced 1.61, slow 1.55). The slow group has only 7 teams.

## 6. Betting odds

### How often the favourite wins

In [25]:
query = """
SELECT
    League.name AS league,
    COUNT(*) AS matches_with_odds,
    ROUND(100.0 * SUM((Match.B365H <= Match.B365A AND Match.home_team_goal > Match.away_team_goal)
                   OR (Match.B365A <  Match.B365H AND Match.away_team_goal > Match.home_team_goal)) / COUNT(*), 1) AS favourite_won_pct,
    ROUND(100.0 * SUM((Match.B365H <= Match.B365A AND Match.home_team_goal < Match.away_team_goal)
                   OR (Match.B365A <  Match.B365H AND Match.away_team_goal < Match.home_team_goal)) / COUNT(*), 1) AS upset_pct
FROM Match
JOIN League ON Match.league_id = League.id
WHERE Match.B365H IS NOT NULL AND Match.B365A IS NOT NULL
GROUP BY League.name
ORDER BY favourite_won_pct DESC
"""

run_query(query)

,league,matches_with_odds,favourite_won_pct,upset_pct
0,Spain LIGA BBVA,3039,56.2,20.6
1,Netherlands Eredivisie,2445,55.6,20.7
2,Portugal Liga ZON Sagres,2044,54.9,19.0
3,Italy Serie A,3011,53.3,20.3
4,England Premier League,3040,53.3,20.9
5,Belgium Jupiler League,1706,52.9,22.5
6,Germany 1. Bundesliga,2447,51.5,24.1
7,Scotland Premier League,1824,50.8,24.7
8,France Ligue 1,3036,50.5,21.2


**Finding:** The favourite wins most often in Spain (56.2%) and least in France (50.5%). The underdog wins most often in Scotland (24.7%) and Germany (24.1%). Poland and Switzerland have no odds.

### Biggest home upsets

In [26]:
query = """
SELECT
    League.name AS league,
    substr(Match.date, 1, 10) AS match_date,
    HomeTeam.team_long_name AS home_team,
    AwayTeam.team_long_name AS away_team,
    Match.home_team_goal || ' - ' || Match.away_team_goal AS score,
    Match.B365H AS home_odds,
    Match.B365A AS away_odds
FROM Match
JOIN League ON Match.league_id = League.id
JOIN Team AS HomeTeam ON Match.home_team_api_id = HomeTeam.team_api_id
JOIN Team AS AwayTeam ON Match.away_team_api_id = AwayTeam.team_api_id
WHERE Match.home_team_goal > Match.away_team_goal
ORDER BY Match.B365H DESC
LIMIT 10
"""

run_query(query)

,league,match_date,home_team,away_team,score,home_odds,away_odds
0,Spain LIGA BBVA,2016-05-08,Levante UD,Atlético Madrid,2 - 1,17,1.20
1,Spain LIGA BBVA,2014-05-04,Levante UD,Atlético Madrid,2 - 0,15,1.22
2,Spain LIGA BBVA,2011-11-26,Getafe CF,FC Barcelona,1 - 0,13,1.20
3,Spain LIGA BBVA,2011-09-18,Levante UD,Real Madrid CF,1 - 0,13,1.20
4,Spain LIGA BBVA,2014-03-08,Real Valladolid,FC Barcelona,1 - 0,12,1.20
5,Netherlands Eredivisie,2015-05-17,FC Dordrecht,Ajax,2 - 1,11,1.25
6,Portugal Liga ZON Sagres,2012-01-29,Gil Vicente FC,FC Porto,3 - 1,11,1.29
7,Portugal Liga ZON Sagres,2013-11-30,Académica de Coimbra,FC Porto,1 - 0,11,1.33
8,England Premier League,2009-09-26,Wigan Athletic,Chelsea,3 - 1,10,1.33
9,England Premier League,2014-03-29,Crystal Palace,Chelsea,1 - 0,10,1.40


**Finding:** Levante beat Atletico Madrid 2-1 in 2016 with home odds of 17; Spanish matches take all top 5 places.

### Biggest away upsets

In [27]:
query = """
SELECT
    League.name AS league,
    substr(Match.date, 1, 10) AS match_date,
    HomeTeam.team_long_name AS home_team,
    AwayTeam.team_long_name AS away_team,
    Match.home_team_goal || ' - ' || Match.away_team_goal AS score,
    Match.B365H AS home_odds,
    Match.B365A AS away_odds
FROM Match
JOIN League ON Match.league_id = League.id
JOIN Team AS HomeTeam ON Match.home_team_api_id = HomeTeam.team_api_id
JOIN Team AS AwayTeam ON Match.away_team_api_id = AwayTeam.team_api_id
WHERE Match.away_team_goal > Match.home_team_goal
ORDER BY Match.B365A DESC
LIMIT 10
"""

run_query(query)

,league,match_date,home_team,away_team,score,home_odds,away_odds
0,Spain LIGA BBVA,2010-09-11,FC Barcelona,Hércules Club de Fútbol,0 - 2,1.06,34
1,England Premier League,2011-12-31,Manchester United,Blackburn Rovers,2 - 3,1.13,26
2,Portugal Liga ZON Sagres,2016-04-04,FC Porto,Tondela,0 - 1,1.10,23
3,Germany 1. Bundesliga,2016-03-02,FC Bayern Munich,1. FSV Mainz 05,1 - 2,1.13,21
4,Spain LIGA BBVA,2015-02-21,FC Barcelona,Málaga CF,0 - 1,1.11,21
5,Portugal Liga ZON Sagres,2016-02-07,FC Porto,FC Arouca,1 - 2,1.17,19
6,Scotland Premier League,2015-12-19,Celtic,Motherwell,1 - 2,1.13,19
7,Spain LIGA BBVA,2014-02-01,FC Barcelona,Valencia CF,2 - 3,1.13,19
8,England Premier League,2014-08-30,Manchester City,Stoke City,0 - 1,1.18,18
9,England Premier League,2008-09-27,Arsenal,Hull City,1 - 2,1.20,17


**Finding:** Hercules won 2-0 at FC Barcelona in 2010 with away odds of 34; Blackburn won 3-2 at Manchester United with odds of 26.